In [ ]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

from datetime import datetime
import sys
import os
import warnings
from pathlib import Path
from typing import Any, Tuple, List, Dict
from dotmap import DotMap

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objs as go
import statsmodels.api as sm
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

import datajoint as dj
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import acquisition, streams
from swc.aeon.io import api as aeon_api
from aeon.schema.schemas import social02

cwd = os.getcwd()
project_path = os.path.join(cwd, "ProjectAeon", "aeon_scratchpad", "aeon_analysis", "aeon_methods_paper")
sys.path.append(project_path)
from data_io_utils import save_all_experiment_data, load_data_from_parquet

# Definitions

In [ ]:
data_dir = Path("/ceph/aeon/aeon/code/scratchpad/methods_paper_data")
os.makedirs(data_dir, exist_ok=True)
cm2px = 5.2  # 1 cm = 5.2 px roughly in aeon arenas

In [ ]:
experiments = [
    {"name": "social0.2-aeon3", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "postsocial_start": '2024-02-25 17:00:00', "postsocial_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 17:00:00', "social_end": '2024-02-23 12:00:00', "postsocial_start": '2024-02-25 18:00:00', "postsocial_end": '2024-03-02 13:00:00'},
    {"name": "social0.3-aeon3", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 13:00:00', "social_start": '2024-06-25 11:00:00', "social_end": '2024-07-06 13:00:00', "postsocial_start": '2024-07-07 16:00:00', "postsocial_end": '2024-07-14 14:00:00'},
    {"name": "social0.3-aeon4", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 14:00:00', "social_start": '2024-06-19 12:00:00', "social_end": '2024-07-03 14:00:00', "postsocial_start": '2024-07-04 11:00:00', "postsocial_end": '2024-07-13 12:00:00'},
    {"name": "social0.4-aeon3", "presocial_start": '2024-08-16 17:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 11:00:00', "social_end": '2024-09-09 13:00:00', "postsocial_start": '2024-09-09 18:00:00', "postsocial_end": '2024-09-22 16:00:00'},
    {"name": "social0.4-aeon4", "presocial_start": '2024-08-16 15:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 10:00:00', "social_end": '2024-09-09 01:00:00', "postsocial_start": '2024-09-09 15:00:00', "postsocial_end": '2024-09-22 16:00:00'}
]
experiment = experiments[0]

# Load data

In [ ]:
def load_experiment_data(experiment, data_dir, periods=['presocial', 'social', 'postsocial'], 
                        data_types=['rfid', 'position'], trim_days=None):
    """
    Load all data types for specified periods of an experiment.
    
    Parameters:
    - experiment: experiment dict with period start/end times
    - periods: list of periods to load
    - data_types: list of data types to load
    - data_dir: directory containing data files
    - trim_days: Optional number of days to trim from start (None = no trim)
    
    Returns:
    - Dictionary containing dataframes for each period/data type combination
    """
    
    result = {}
    
    for period in periods:
        for data_type in data_types:
            print(f"Loading {period} {data_type} data...")
            
            # Load data
            df = load_data_from_parquet(
                experiment_name=experiment["name"],
                period=period,
                data_type=data_type,
                data_dir=data_dir,
                set_time_index=(data_type == 'position')
            )
            
            # Trim if requested
            if trim_days is not None and len(df) > 0:
                if data_type == 'rfid':
                    start_time = df['chunk_start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['chunk_start'] < end_time]
                else:  # position data
                    start_time = df.index.min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df.loc[df.index < end_time]
                
                print(f"  Trimmed to {trim_days} days: {len(df)} records")
            
            # Store in result
            key = f"{period}_{data_type}"
            result[key] = df
            
            # For position data, handle duplicates
            if data_type == 'position' and len(df) > 0:
                original_len = len(df)
                df = df.reset_index()
                df = df.drop_duplicates(subset=['time', 'identity_name'], keep='first')
                df = df.set_index('time')
                result[key] = df
                if len(df) < original_len:
                    print(f"  Removed duplicates: {original_len} -> {len(df)}")
    
    return result

# Load all periods for experiment
data = load_experiment_data(
    experiment=experiment,
    data_dir=data_dir,
    periods=['presocial', 'social', 'postsocial'],
    data_types=['rfid', 'position'],
    trim_days=4  # Optional: trim to 4 days
)

# Access data
presocial_rfid_df = data['presocial_rfid']
presocial_position_df = data['presocial_position']
social_rfid_df = data['social_rfid']
social_position_df = data['social_position']
postsocial_rfid_df = data['postsocial_rfid']
postsocial_position_df = data['postsocial_position']

In [ ]:
# RFID patches 1 and 2 are swapped in the presocial period of the social0.2-aeon3 experiment
if experiment["name"] == "social0.2-aeon3":
    # First, create a temporary placeholder to avoid overwriting during the swap
    presocial_rfid_df['rfid_reader_name'] = presocial_rfid_df['rfid_reader_name'].replace({
        'Patch1Rfid': 'TEMP_PLACEHOLDER',
        'Patch2Rfid': 'Patch1Rfid'
    })
    # Now replace the placeholder with Patch2Rfid
    presocial_rfid_df['rfid_reader_name'] = presocial_rfid_df['rfid_reader_name'].replace({
        'TEMP_PLACEHOLDER': 'Patch2Rfid'
    })
    # Verify the swap worked
    print(presocial_rfid_df["rfid_reader_name"].unique())

In [ ]:
acquisition_computer = experiment["name"].split("-")[1].upper()
social_name = experiment["name"].split("-")[0]
metadata_root = Path(f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}")
metadata_reader = social02.Metadata
metadata = aeon_api.load(metadata_root, metadata_reader)['metadata'].iloc[0]
rfid_devices_loc = DotMap({key: metadata.Devices[key].Location for key in metadata.Devices.keys() if 'rfid' in key.lower()})
rfid_devices_loc

# Match RFID and Pose rows based on time and position

In [ ]:
def match_rfid_to_pose(rfid_df, pose_df, rfid_devices_loc, cm2px, tolerance_ms=10):
    """
    Match RFID timestamps to nearest pose timestamps within tolerance.
    Returns dataframe with matched data, including unmatched RFID rows.
    """
    # Explode RFID data to have one row per timestamp/identity
    rfid_exploded = (
        rfid_df[['rfid_reader_name', 'timestamps', 'rfid']]
        .explode(['timestamps', 'rfid'])
        .rename(columns={'timestamps': 'rfid_time', 'rfid': 'rfid_identity'})
    )
    rfid_exploded['rfid_time'] = pd.to_datetime(rfid_exploded['rfid_time'])
    rfid_exploded['row_id'] = range(len(rfid_exploded))
    
    # Drop rows with null time values before merging
    rfid_valid = rfid_exploded.dropna(subset=['rfid_time']).copy()
    
    # Get RFID reader coordinates
    reader_coords = {}
    for name, loc in rfid_devices_loc.items():
        if (name.startswith('_') or 
            name.startswith('*') or 
            callable(getattr(rfid_devices_loc, name, None)) or
            name in ['to_pandas', 'to_dict', 'toDict', 'empty', 'copy']):
            continue
        if hasattr(loc, 'X') and hasattr(loc, 'Y'):
            reader_coords[name] = (float(loc.X), float(loc.Y))
    
    # Add reader coordinates to rfid_valid
    rfid_valid['rfid_x'] = rfid_valid['rfid_reader_name'].map(lambda x: reader_coords.get(x, (None, None))[0])
    rfid_valid['rfid_y'] = rfid_valid['rfid_reader_name'].map(lambda x: reader_coords.get(x, (None, None))[1])
    
    # Prepare pose data for efficient merging
    pose_subset = (
        pose_df[['identity_name', 'identity_likelihood', 'x', 'y', 'likelihood']]
        .reset_index()
        .rename(columns={
            'identity_name': 'pose_identity', 
            'time': 'pose_time', 
            'x': 'pose_x', 
            'y': 'pose_y',
            'identity_likelihood': 'pose_identity_likelihood',
            'likelihood': 'pose_likelihood'
        })
    )
    
    # Merge RFID with all possible pose identities to find all candidates
    unique_identities = pose_subset['pose_identity'].unique()
    rfid_expanded = pd.concat([
        rfid_valid.assign(pose_identity=identity) 
        for identity in unique_identities
    ])
    
    # Use merge_asof to find nearest timestamps within tolerance
    merged = pd.merge_asof(
        rfid_expanded.sort_values('rfid_time'),
        pose_subset.sort_values('pose_time'),
        left_on='rfid_time',
        right_on='pose_time',
        by='pose_identity',
        direction='nearest',
        tolerance=pd.Timedelta(f'{tolerance_ms}ms')
    )
    
    # Calculate spatial distances between reader and animal positions in cm
    merged['rfid_pose_distance'] = np.sqrt(
        (merged['pose_x'] - merged['rfid_x'])**2 + 
        (merged['pose_y'] - merged['rfid_y'])**2
    ) / cm2px
    
    # Keep only the closest match for each RFID detection (by distance)
    idx_closest = merged.groupby('row_id')['rfid_pose_distance'].idxmin()
    valid_idx = idx_closest.dropna()
    
    # Start with all rfid_valid rows
    result = rfid_valid.set_index('row_id')

    # Create empty dataframe with correct structure from merged
    empty_template = merged[['pose_time', 'pose_identity', 'pose_identity_likelihood', 
                            'pose_x', 'pose_y', 'pose_likelihood', 'rfid_pose_distance']].iloc[:0]
    result = result.join(empty_template)
    
    # Update with matched data where available
    if len(valid_idx) > 0:
        matched_data = merged.loc[valid_idx].set_index('row_id')
        result.update(matched_data)
    
    result = result.reset_index()
    
    # Reorder columns
    column_order = [
        'rfid_time', 'rfid_reader_name', 'rfid_identity', 'rfid_x', 'rfid_y',
        'pose_time', 'pose_identity', 'pose_identity_likelihood', 'pose_x', 'pose_y', 
        'pose_likelihood', 'rfid_pose_distance'
    ]
    
    # Select columns that exist in the result
    existing_columns = [col for col in column_order if col in result.columns]
    final_result = result[existing_columns].reset_index(drop=True)
    
    return final_result

In [ ]:
pre_post_social_rfid_df = pd.concat([presocial_rfid_df, postsocial_rfid_df], ignore_index=True)
pre_post_social_position_df = pd.concat([presocial_position_df, postsocial_position_df], ignore_index=False)
pre_post_social_matched_df = match_rfid_to_pose(pre_post_social_rfid_df, pre_post_social_position_df, rfid_devices_loc, tolerance_ms=10, cm2px=cm2px)
print(pre_post_social_matched_df["rfid_time"].isna().sum()) # should be 0
display(pre_post_social_matched_df)
social_matched_df = match_rfid_to_pose(social_rfid_df, social_position_df, rfid_devices_loc, tolerance_ms=10, cm2px=cm2px)
display(social_matched_df)

# RFID readers' range

In [ ]:
# Calculate RFID reader ranges
quantiles = [0.95, 0.99, 1.0]
reader_ranges = (
    pre_post_social_matched_df
    .dropna(subset=['rfid_pose_distance'])
    .groupby('rfid_reader_name')['rfid_pose_distance']
    .agg(['median', 'mean', ('p95', lambda x: x.quantile(0.95)), 
          ('p99', lambda x: x.quantile(0.99)), 'max'])
    .round(1)
)

print("RFID Reader Ranges (cm):")
print(reader_ranges)

In [ ]:
# # Optional debugging
# import swc.aeon.io.reader
# import swc.aeon.io.api
# from aeon.schema.schemas import exp02
# from aeon.analysis.movies import gridframes
# from aeon.io.video import frames
# from aeon.dj_pipeline.analysis.block_analysis import *
# import plotly.express as px
# # Plot the data on the corresponding video frame
# idx = 1
# fps = 50
# time = pre_post_social_matched_df.iloc[idx]['pose_time']
# df_to_plot = pre_post_social_matched_df.iloc[idx:idx+1]
# display(df_to_plot)
# root = Path(f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}")
# vid_data = swc.aeon.io.api.load(root, exp02.CameraTop.Video, start=time, end=time+pd.Timedelta(seconds=1/fps/2))
# vid_data = vid_data[time:time] # idk why this is necessary by for some reason the video data is loading more than just the ts between time and time+1/fps/2
# fig = px.imshow(gridframes(list(frames(vid_data)), width=1440, height=1080, shape=1))
# # Add scatter plot with larger marker size
# scatter_trace = px.scatter(df_to_plot, x='pose_x', y='pose_y', size_max=15).data[0]
# scatter_trace2 = px.scatter(df_to_plot, x='rfid_x', y='rfid_y', size_max=15).data[0]
# scatter_trace.marker.size = 6  # Increase marker size
# scatter_trace.marker.color = 'red'  # Make markers more visible
# fig.add_trace(scatter_trace)
# fig.add_trace(scatter_trace2)
# # Update layout to make plot bigger and adjust margins
# fig.update_layout(
#     width=1200,
#     height=1000,
#     margin=dict(l=20, r=20, t=20, b=20),  # Reduce margins to use more space
#     showlegend=False
# )
# # Show the figure
# fig.show()

In [ ]:
# Calculate RFID reader ranges with social data 
# (this should be less accurate because a RFID reading could potentially be matched to the wrong subject if only 1 subject is detected by SLEAP)
quantiles = [0.95, 0.99, 1.0]
reader_ranges = (
    social_matched_df
    .dropna(subset=['rfid_pose_distance'])
    .groupby('rfid_reader_name')['rfid_pose_distance']
    .agg(['median', 'mean', ('p95', lambda x: x.quantile(0.95)), 
          ('p99', lambda x: x.quantile(0.99)), 'max'])
    .round(1)
)

print("RFID Reader Ranges (cm):")
print(reader_ranges)

In [ ]:
# Calculate RFID reader temporal accuracy
rfid_freq = 2 # Hz

# TODO

In [ ]:
# Plot RFID reader reads

# Group data by rfid_reader_name
reader_names = social_matched_df['rfid_reader_name'].unique()

# Create a figure
time = pd.Timestamp('2024-02-03 16:28:17')
root = Path(f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}")
vid_data = swc.aeon.io.api.load(root, exp02.CameraTop.Video, start=time, end=time+pd.Timedelta(seconds=1/fps/2))
vid_data = vid_data[time:time] # idk why this is necessary by for some reason the video data is loading more than just the ts between time and time+1/fps/2
fig = px.imshow(gridframes(list(frames(vid_data)), width=1440, height=1080, shape=1))

# 1) get unique readers & their static positions
readers_df = (
    social_matched_df
    .groupby('rfid_reader_name')
    .agg(rfid_x=('rfid_x','first'),
         rfid_y=('rfid_y','first'))
    .reset_index()
)

reader_names = readers_df['rfid_reader_name'].tolist()
n = len(reader_names)

# 2) pick a qualitative palette with enough distinct colors
colors = px.colors.qualitative.Plotly
# if you have more readers than the palette length, you can cycle:
colors = [colors[i % len(colors)] for i in range(n)]

# 3) loop over readers and add scatter traces
for i, reader in enumerate(reader_names):
    col = colors[i]
    # filter out rows for this reader
    sub = social_matched_df[social_matched_df['rfid_reader_name'] == reader]

    # a) subject detections
    fig.add_trace(
        go.Scattergl(
            x=sub['pose_x'],
            y=sub['pose_y'],
            mode='markers',
            marker=dict(
                color=col,
                size=6,
                opacity=0.3
            ),
            name=f"{reader} detections"
        )
    )

    # b) reader static position
    rx = readers_df.loc[readers_df['rfid_reader_name']==reader, 'rfid_x'].item()
    ry = readers_df.loc[readers_df['rfid_reader_name']==reader, 'rfid_y'].item()
    col = colors[i]

    # darker but still saturated
    dark_col = px.colors.label_rgb(px.colors.hex_to_rgb(col))

    fig.add_trace(
        go.Scattergl(
            x=[rx],
            y=[ry],
            mode='markers+text',
            marker=dict(
                symbol='x',   
                size=10,    
                color="black",
                opacity=1.0,
            ),
            text=[reader],
            textposition='top center',
            textfont=dict(
                size=14,
                family='Arial Black',
                color="black"
            ),
            showlegend=False
        )
    )

# now update layout as you already have
fig.update_layout(
    width=1200,
    height=1000,
    margin=dict(l=20, r=20, t=20, b=20),
)

fig.show()


# Assess SLEAP accuracy

In [ ]:
# Extract date from rfid_time
social_matched_df['date'] = pd.to_datetime(social_matched_df['rfid_time']).dt.date

# Group by date and calculate metrics
results = []
for date, group in social_matched_df.groupby('date'):
    # Total rows for this day
    total = len(group)
    
    # Rows where SLEAP made a prediction
    predicted = group['pose_identity'].notna().sum()
    
    # Rows where prediction matches ground truth (only consider rows with predictions)
    correct = ((group['pose_identity'] == group['rfid_identity']) & 
               group['pose_identity'].notna()).sum()
    
    # Calculate metrics
    id_accuracy = correct / predicted if predicted > 0 else 0
    missed_rate = (total - predicted) / total
    
    results.append({
        'date': date,
        'id_accuracy': id_accuracy,
        'missed_rate': missed_rate
    })

# Convert to DataFrame for plotting
results_df = pd.DataFrame(results).sort_values('date')

# Create a single figure with both metrics
fig = go.Figure()

# Add ID accuracy trace
fig.add_trace(
    go.Scatter(
        x=results_df['date'], 
        y=results_df['id_accuracy'],
        mode='lines+markers',
        name='ID Accuracy',
        line=dict(color='blue')
    )
)

# Add missed prediction rate trace
fig.add_trace(
    go.Scatter(
        x=results_df['date'], 
        y=results_df['missed_rate'],
        mode='lines+markers',
        name='Missed Prediction Rate',
        line=dict(color='red')
    )
)

# Update layout
fig.update_layout(
    height=500,
    width=900,
    title="SLEAP Performance Metrics by Day",
    xaxis_title="Date",
    yaxis_title="Rate",
    yaxis=dict(range=[0, 1], tickformat='.0%'),
    showlegend=True
)

# Show figure
fig.show()

# Print summary statistics
display(results_df)
print(f"Average ID Accuracy: {results_df['id_accuracy'].mean():.2%}")
print(f"Average Missed Prediction Rate: {results_df['missed_rate'].mean():.2%}")